In [1]:
import sys
import os
import pandas as pd
import time

In [2]:
measured = pd.read_csv('Manual_Foraging_Events_Observation.csv')

In [4]:
# load from final csv
predicted = pd.read_csv('CVPR_Evaluation_Results_v3.csv')

predicted = predicted[['action', 'nest', 'frame_number',"video","timestamp","filename"]].copy()

In [5]:
predicted.dropna(inplace=True)

In [6]:
#predicted = 
predicted.reset_index(drop=True, inplace=True)

In [7]:
predicted['video'] = predicted['video'].apply(lambda x: x.replace('.mp4', ''))

In [8]:
predicted['timestamp'] = predicted['timestamp'].astype(str)
predicted['timestamp'] = predicted['timestamp'].apply(lambda x: x.split(' ')[1])

In [9]:
measured = measured[['video','action','nest','timestamp']].dropna()

In [10]:
from datetime import time
def getTimestamp1(txt):
    hr, mn, s = txt.split(':')
    #return timedelta(hours=int(hr), minutes=int(mn), seconds=int(s))
    return time(int(hr), int(mn), int(s))
measured['timestamp'] = measured['timestamp'].apply(getTimestamp1)

In [11]:
def getTimestamp2(txt):
    hr, mn, s = txt.split(':')
    #return timedelta(hours=int(hr), minutes=int(mn), seconds=int(s))
    return time(int(hr), int(mn), int(float(s)))
predicted['timestamp'] = predicted['timestamp'].apply(getTimestamp2)


In [12]:
measured['site'] = measured['video'].apply(lambda x: x.split('_')[0])
measured['hour'] = measured['timestamp'].apply(lambda x: x.hour)

predicted['site'] = predicted['video'].apply(lambda x: x.split('_')[0])
predicted['hour'] = predicted['timestamp'].apply(lambda x: x.hour)

In [13]:
# filter measured based on videos in predicted
videos = predicted.video.unique().tolist()
videos = [v.replace('.mp4', '') for v in videos]
measured_temp = measured[measured['video'].isin(videos)]

In [14]:
len(measured_temp)

300

In [15]:
measured.groupby('video').count()

,action,nest,timestamp,site,hour
video,,,,,
mendels_2024-04-30_09_00_00,10,10,10,10,10
mendels_2024-04-30_09_10_01,16,16,16,16,16
mendels_2024-04-30_09_20_00,9,9,9,9,9
mendels_2024-04-30_09_30_00,18,18,18,18,18
mendels_2024-04-30_09_40_01,5,5,5,5,5
mendels_2024-05-08_15_00_00,23,23,23,23,23
mendels_2024-05-08_15_30_00,26,26,26,26,26
mendels_2024-05-08_15_50_00,23,23,23,23,23
mendels_2024-05-23_12_00_00,27,27,27,27,27


In [16]:
predicted.groupby('video').count()

,action,nest,frame_number,timestamp,filename,site,hour
video,,,,,,,
mendels_2024-04-30_09_00_00,15,15,15,15,15,15,15
mendels_2024-04-30_09_10_01,12,12,12,12,12,12,12
mendels_2024-04-30_09_20_00,9,9,9,9,9,9,9
mendels_2024-04-30_09_30_00,20,20,20,20,20,20,20
mendels_2024-04-30_09_40_01,8,8,8,8,8,8,8
mendels_2024-05-08_15_00_00,18,18,18,18,18,18,18
mendels_2024-05-08_15_30_00,21,21,21,21,21,21,21
mendels_2024-05-08_15_50_00,31,31,31,31,31,31,31
mendels_2024-05-23_12_00_00,25,25,25,25,25,25,25


In [17]:
measured_temp.reset_index(drop=True, inplace=True)
predicted.reset_index(drop=True, inplace=True)

In [18]:
class Action:
    def __init__(self, action, timestamp, nest, video):
        self.action = action
        self.timestamp = timestamp
        self.nest = int(nest)
        self.video = video

    def getAction(self):
        return self.action
    
    def getTimestamp(self):
        return self.timestamp
    
    def getNest(self):
        return self.nest
    
    def getVideo(self):
        return self.video

def getActions(df):
    actions = []
    for i in range(len(df)):
        action = Action(df['action'][i], df['timestamp'][i], df['nest'][i], df['video'][i])
        actions.append(action)
    return actions

from datetime import datetime

def time_difference(time1, time2):
    # Convert the time strings to datetime objects
    date_today = datetime.today().date()
    datetime1 = datetime.combine(date_today, time1)
    datetime2 = datetime.combine(date_today, time2)

    # Calculate the difference
    time_difference = datetime1 - datetime2

    # Get the difference in seconds
    difference_in_seconds = time_difference.total_seconds()

    return abs(difference_in_seconds)

def isActionInActions(action, actions):
    for act in actions:

        if action.action == act.action and time_difference(action.timestamp, act.timestamp) < 3 and action.video == act.video and action.nest == act.nest:
            return True
        
    return False

In [19]:
measured_actions = getActions(measured_temp)

In [20]:
predicted_actions = getActions(predicted)


In [21]:
def calculateTruePositives(measured_actions, predicted_actions):
    tp = 0
    objs = []
    for action in predicted_actions:
        if isActionInActions(action, measured_actions):
            tp += 1
            objs.append(action)
    return tp, objs

tp, tp_obj = calculateTruePositives(measured_actions, predicted_actions)
print(tp)

192


In [22]:
tp_df = pd.DataFrame([obj.__dict__ for obj in tp_obj])
tp_df.groupby('action').count()

,timestamp,nest,video
action,,,
Entry,107,107,107
Exit,85,85,85


In [23]:
def calculateFalsePositives(measured_actions, predicted_actions):
    fp = 0
    fp_obj = []
    for action in predicted_actions:
        if not isActionInActions(action, measured_actions):
            fp += 1
            fp_obj.append(action)
    return fp, fp_obj

fp, fp_obj = calculateFalsePositives(measured_actions, predicted_actions)
print(fp)

67


In [24]:
fp_df = pd.DataFrame([obj.__dict__ for obj in fp_obj])
fp_df.groupby('action').count()

,timestamp,nest,video
action,,,
Entry,49,49,49
Exit,18,18,18


In [25]:
def calculateFalseNegatives(measured_actions, predicted_actions):
    fn = 0
    fn_obj = []
    for action in measured_actions:
        if not isActionInActions(action, predicted_actions):
            fn += 1
            fn_obj.append(action)
    return fn, fn_obj

fn, fn_obj = calculateFalseNegatives(measured_actions, predicted_actions)
print(fn)

108


In [26]:
fn_df = pd.DataFrame([obj.__dict__ for obj in fn_obj])
fn_df.groupby('action').count()

,timestamp,nest,video
action,,,
Entry,42,42,42
Exit,66,66,66


In [27]:
import numpy as np

In [28]:
# overall precision
np.mean(tp_df.groupby('video').size() / predicted.groupby('video').size()).tolist()

0.6984108247404532

In [29]:
# precision per video
tp_df.groupby('video').size() / predicted.groupby('video').size()

video
mendels_2024-04-30_09_00_00    0.400000
mendels_2024-04-30_09_10_01    0.833333
mendels_2024-04-30_09_20_00    0.666667
mendels_2024-04-30_09_30_00    0.750000
mendels_2024-04-30_09_40_01    0.500000
mendels_2024-05-08_15_00_00    0.666667
mendels_2024-05-08_15_30_00    0.714286
mendels_2024-05-08_15_50_00    0.612903
mendels_2024-05-23_12_00_00    0.880000
mendels_2024-05-23_12_40_00    0.862745
mendels_2024-05-23_18_20_01    0.795918
dtype: float64

In [30]:
# recall per video
tp_df.groupby('video').size() / measured_temp.groupby('video').size()

video
mendels_2024-04-30_09_00_00    0.600000
mendels_2024-04-30_09_10_01    0.625000
mendels_2024-04-30_09_20_00    0.666667
mendels_2024-04-30_09_30_00    0.833333
mendels_2024-04-30_09_40_01    0.800000
mendels_2024-05-08_15_00_00    0.521739
mendels_2024-05-08_15_30_00    0.576923
mendels_2024-05-08_15_50_00    0.826087
mendels_2024-05-23_12_00_00    0.814815
mendels_2024-05-23_12_40_00    0.666667
mendels_2024-05-23_18_20_01    0.506494
dtype: float64

In [31]:
# overall recall
np.mean(tp_df.groupby('video').size() / measured_temp.groupby('video').size()).tolist()

0.6761567410776897